In [ ]:
!pip install -r requirements.txt

In [ ]:
import wandb
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from scipy.stats import ks_2samp, chi2_contingency
import yaml
import random
import matplotlib.pyplot as plt
import kagglehub
import shutil
import os

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
from key import KEY
API_KEY=KEY
wandb.login(API_KEY)

In [ ]:
config = {
    "data": {
        "raw_path": "data/raw/dataset.csv"
    }
}

In [ ]:
path = kagglehub.dataset_download("lovishbansal123/nasa-asteroids-classification")
print(f'Baixado para: {path}')

destination = os.path.join(os.getcwd(), "dataset")

shutil.copytree(path, destination, dirs_exist_ok=True)

df_raw = pd.read_csv("/home/joaovitor/Documents/Faculdade/1°Semestre/Tópicos avançados em Inteligência Artificial A/Avaliação da 1° Unidade/dataset/nasa.csv", low_memory=False)

print(f"Linhas: {df_raw.shape}")
print(f"Colunas: {list(df_raw.columns)}")

df_raw.head()

In [ ]:
wandb.init(
    project="MLOps-Work",
    job_type="load_raw",
    name="load_raw"
)

artifact = wandb.Artifact(
    name="raw_data",
    type="dataset",
    description="nasa-asteroids-classification"
)
temp_path = "temp_raw.csv"
df_raw.to_csv(temp_path, index=False)
artifact.add_file(temp_path)

wandb.log_artifact(artifact)

wandb.summary["rows"] = len(df_raw)
wandb.summary["columns"] = list(df_raw.columns)

wandb.finish()

In [ ]:
# Transformando o True em 1 e False em 0.
df_raw['Hazardous'] = df_raw['Hazardous'].replace({True: 1, False: 0})

df_raw.head()

# Removendo a coluna: "Equinox" e "Orbiting body"

df = df_raw.drop(['Equinox', 'Orbiting Body'], axis=1)
df.head()

In [ ]:
def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates()
    removidas = before - len(df)
    print(f'Removidas: {removidas} linhas duplicadas'
          f"({removidas / before:.1%} of dataset)")
    return df
df_clean = remove_duplicates(df_raw)

In [ ]:
def handle_missing_values(df, strategy='mean', threshold=0.5):
    missing_frac = df.isnull().mean()
    cols_to_drop = missing_frac[missing_frac > threshold].index.tolist()
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped columns: {cols_to_drop}")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    imputer = SimpleImputer(strategy=strategy)
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
    cat_cols = df.select_dtypes(include=['object']).columns
    df[cat_cols] = df[cat_cols].fillna('missing')
    return df

df_clean = remove_duplicates(df_raw)
df_clean = handle_missing_values(df_clean, 
                                 strategy=config['data']['imputation_strategy'],
                                 threshold=config['data']['missing_threshold'])
print(f"Dataset limpo: {len(df_clean)} amostras")

wandb.init(project="MLOps-Work", job_type="clean_data", name="clean_data")
artifact = wandb.Artifact("clean_data", type="dataset")
temp_path = "temp_clean.csv"
df_clean.to_csv(temp_path, index=False)
artifact.add_file(temp_path)
wandb.log_artifact(artifact)
wandb.summary["rows"] = len(df_clean)
wandb.finish()

In [ ]:
wandb.init(
    project="MLOps-Work",
    job_type="clean_data",
    name="clean_data"
)

artifact = wandb.Artifact(
    name="clean_data",
    type="dataset",
    description="nasa-asteroids-classification after remove duplicates and missing values"
)
temp_path = "temp_clean.csv"
df_raw.to_csv(temp_path, index=False)
artifact.add_file(temp_path)

wandb.log_artifact(artifact)

wandb.summary["rows"] = len(df_clean)
wandb.summary["dropped_columns"] = df_clean.shape[1]

wandb.finish()